<a href="https://colab.research.google.com/github/technologyhamed/LLMAttack/blob/main/HMM/step1_apt_sequence_builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# گام ۱ — ساخت دیتاست `Apt-sequence.json` از فایل MITRE ATT&CK

**هدف:** تولید دیتاست پیش‌بینی تکنیک بعدی (`next_technique_prediction`) با واحد دنباله = رتبهٔ تاکتیک (`tactic_rank`)، دقیقاً مطابق قالب نمونهٔ ارسالی.

**منبع:** `enterprise-attack-v19_2.xlsx` (شیت‌های `techniques`, `tactics`, `groups`, `campaigns`, `relationships`, `software`)

## منطق ساخت نمونه‌ها (مطابق قالب نمونه)
1. برای هر «کمپین» (گروه APT یا کمپین ATT&CK) تکنیک‌های استفاده‌شده جمع می‌شود و هر تکنیک به تاکتیک/رتبهٔ آن نگاشت می‌شود.
2. تکنیک‌ها بر اساس رتبهٔ تاکتیک (ترتیب Kill-Chain) گروه‌بندی می‌شوند.
3. برای هر جفت رتبهٔ متوالی `(r_i → r_{i+1})`: ورودی = **همهٔ** تکنیک‌های رتبه‌های ≤ `r_i` (به‌ترتیب رتبه)، و به ازای **هر** تکنیک رتبهٔ `r_{i+1}` یک نمونه با آن `target` ساخته می‌شود.
4. **پنجرهٔ لغزان (اختیاری):** با `WINDOW_SIZE = N` فقط N تکنیک آخر تاریخچه به‌عنوان ورودی نگه داشته می‌شود. مقدار پیش‌فرض `None` است (تاریخچهٔ کامل، مطابق نمونهٔ ارسالی).

## تصمیم‌های طراحی
- **رتبه‌ها:** `reconnaissance=0 … impact=14` بر اساس ترتیب ماتریس نسخهٔ v19 (در این نسخه `defense-evasion` به `stealth` و `defense-impairment` تقسیم شده است؛ پس رتبه‌ها با نمونهٔ قدیمی کمی فرق دارد، مثلاً C2 = 12 به‌جای 11).
- **تکنیک چندتاکتیکی:** پیش‌فرض `first` (اولین تاکتیک در ترتیب Kill-Chain). حالت `expand` هم پیاده شده است.
- **فیلدهای `object` و `target_area`:** در فایل اکسل ATT&CK فیلد مستقل ندارند؛ با کلیدواژه از توضیحات گروه/کمپین استخراج می‌شوند (heuristic) و در صورت نبود اطلاعات `Unknown` می‌شوند.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, re, collections
from pathlib import Path
import pandas as pd

# ----------------------------- تنظیمات -----------------------------
XLSX_CANDIDATES = [
    "/content/enterprise-attack-v19_2.xlsx",
    "enterprise-attack-v19_2.xlsx",
]
XLSX_PATH = next((p for p in XLSX_CANDIDATES if Path(p).exists()), None)
assert XLSX_PATH, "فایل اکسل پیدا نشد؛ مسیر را در XLSX_CANDIDATES اصلاح کنید."

OUTPUT_DIR  = Path("/content/user-data/outputs") if Path("/content/user-data/outputs").exists() else Path(".")
OUTPUT_PATH = OUTPUT_DIR / "Apt-sequence.json"

INCLUDE_GROUPS              = True    # گروه‌های APT (مثل APT-C-36)
INCLUDE_ATTACK_CAMPAIGNS    = True    # کمپین‌های رسمی ATT&CK (C00xx)
INCLUDE_SOFTWARE_TECHNIQUES = False   # افزودن تکنیک‌های نرم‌افزارهای مورداستفادهٔ گروه
MULTI_TACTIC_POLICY         = "first" # "first" | "expand"
WINDOW_SIZE                 = None    # None = تاریخچهٔ کامل | عدد N = پنجرهٔ لغزان
MIN_DISTINCT_RANKS          = 2       # حداقل تعداد رتبهٔ متمایز برای معتبر بودن کمپین

print("source:", XLSX_PATH)
print("output:", OUTPUT_PATH)

source: /content/enterprise-attack-v19_2.xlsx
output: Apt-sequence.json


In [ ]:
# ----------------------------- بارگذاری شیت‌ها -----------------------------
xl = pd.ExcelFile(XLSX_PATH)
techniques = xl.parse("techniques")
tactics    = xl.parse("tactics")
groups     = xl.parse("groups")
campaigns  = xl.parse("campaigns")
rels       = xl.parse("relationships")

print({k: v.shape for k, v in dict(techniques=techniques, tactics=tactics, groups=groups,
                                    campaigns=campaigns, relationships=rels).items()})

{'techniques': (697, 17), 'tactics': (15, 9), 'groups': (176, 13), 'campaigns': (56, 17), 'relationships': (20628, 13)}


In [ ]:
# ----------------------------- ترتیب و رتبهٔ تاکتیک‌ها -----------------------------
def slug(name: str) -> str:
    return re.sub(r"\s+", "-", str(name).strip().lower())

# ترتیب ستون‌های ماتریس Enterprise ATT&CK v19
TACTIC_ORDER = [
    "reconnaissance", "resource-development", "initial-access", "execution",
    "persistence", "privilege-escalation", "stealth", "defense-impairment",
    "credential-access", "discovery", "lateral-movement", "collection",
    "command-and-control", "exfiltration", "impact",
]
TACTIC_RANK = {t: i for i, t in enumerate(TACTIC_ORDER)}

# اعتبارسنجی با شیت tactics (اگر نسخهٔ ATT&CK عوض شود اینجا خطا می‌دهد)
sheet_tactics = {slug(n) for n in tactics["name"]}
assert sheet_tactics == set(TACTIC_ORDER), (sheet_tactics ^ set(TACTIC_ORDER))
print(TACTIC_RANK)

{'reconnaissance': 0, 'resource-development': 1, 'initial-access': 2, 'execution': 3, 'persistence': 4, 'privilege-escalation': 5, 'stealth': 6, 'defense-impairment': 7, 'credential-access': 8, 'discovery': 9, 'lateral-movement': 10, 'collection': 11, 'command-and-control': 12, 'exfiltration': 13, 'impact': 14}


In [ ]:
# ----------------------------- نگاشت تکنیک -> [(rank, tactic)] -----------------------------
def build_technique_tactics(df: pd.DataFrame, policy: str = "first") -> dict:
    mapping = {}
    for tid, tac in zip(df["ID"], df["tactics"]):
        if pd.isna(tac):
            continue
        pairs = sorted({(TACTIC_RANK[slug(t)], slug(t)) for t in str(tac).split(",") if slug(t) in TACTIC_RANK})
        if not pairs:
            continue
        mapping[tid] = pairs[:1] if policy == "first" else pairs
    return mapping

TECH_TACTICS = build_technique_tactics(techniques, MULTI_TACTIC_POLICY)
print("techniques with tactic:", len(TECH_TACTICS), "/", len(techniques))
print("T1134 ->", TECH_TACTICS["T1134"])

techniques with tactic: 697 / 697
T1134 -> [(5, 'privilege-escalation')]


In [ ]:
# ----------------------------- استخراج object / target_area از توضیحات (heuristic) -----------------------------
def clean_text(s) -> str:
    if pd.isna(s):
        return ""
    s = re.sub(r"\(Citation:[^)]*\)", " ", str(s))
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)   # [text](url) -> text
    return re.sub(r"\s+", " ", s).strip()

OBJECT_PATTERNS = {
    "Cyber Espionage": r"espionage|intelligence (collection|gathering)|surveillance|\bspy",
    "Financial":       r"financially[- ]motivated|financial gain|financially driven|monetary|ransomware|extortion|cybercrim|criminal|\bfraud|\bprofit",
    "Destruction":     r"destructive|wiper|sabotage|disruptive|disrupt",
    "Hacktivism":      r"hacktivis",
}
REGION_PATTERNS = {
    "Latin America":   r"latin america|south america|central america|colombia|brazil|mexico|argentina|chile|peru|ecuador|venezuela",
    "North America":   r"united states|u\.s\.|north america|canada",
    "Europe":          r"europe|ukrain|poland|germany|france|united kingdom|\buk\b|italy|spain|nato|baltic|nordic",
    "Russia and CIS":  r"russia|belarus|kazakhstan|\bcis\b",
    "Middle East":     r"middle east|israel|iran|saudi|\buae\b|united arab emirates|turkey|iraq|syria|lebanon|palestin|qatar|kuwait",
    "East Asia":       r"china|taiwan|japan|korea|hong kong|mongolia",
    "South Asia":      r"south asia|india|pakistan|bangladesh|sri lanka|nepal",
    "Southeast Asia":  r"southeast asia|vietnam|thailand|indonesia|malaysia|philippines|singapore|myanmar|cambodia|\blaos\b",
    "Africa":          r"africa|nigeria|kenya|egypt|ethiopia|morocco|algeria|tunisia|libya|sudan|ghana",
    "Global":          r"worldwide|global(ly)?\b|across the globe|multiple regions|various countries",
}
SECTOR_PATTERNS = {
    "Government":              r"government|ministr|diplomat|embass|municipal|public sector",
    "Defense":                 r"defen[cs]e|military|armed forces|aerospace",
    "Financial":               r"financial (sector|institution|service|entit|industr|organi)|\bin the financial\b|\bbanks?\b|banking|cryptocurrenc|fintech|insurance",
    "Energy":                  r"energy|electric|power (grid|plant|sector)|\boil\b|\bgas\b|utilit|nuclear",
    "Telecommunications":      r"telecom",
    "Healthcare":              r"health|pharma|medical|hospital|biotech",
    "Education":               r"educat|universit|academi|research (institution|organization)",
    "Manufacturing":           r"manufactur|industrial|automotive|semiconductor",
    "Technology":              r"technology|software|managed service|cloud|supply chain",
    "Media":                   r"(?<!social )media\b|journalis",
    "Critical Infrastructure": r"critical infrastructure|\bics\b|scada|water (utilit|sector|treatment)|transportation|logistic|shipping|aviation|maritime",
    "Retail and Hospitality":  r"retail|hospitality|restaurant|e-?commerce|casino|gaming",
    "NGO and Civil Society":   r"\bngo|non-?profit|human rights|dissident|activist|think tank|civil society",
    "Legal":                   r"law firm|legal",
}
TARGET_SENT = re.compile(r"\btarget(ed|s|ing)?\b|victim|compromis", re.I)

def _match(patterns: dict, text: str) -> list:
    return [k for k, p in patterns.items() if re.search(p, text, re.I)]

def infer_object(desc: str) -> str:
    found = _match(OBJECT_PATTERNS, desc)
    if not found:
        return "Unknown"
    return found[0] if len(found) == 1 else ", ".join(found[:-1]) + " and " + found[-1]

def infer_target_area(desc: str) -> str:
    sents = [s for s in re.split(r"(?<=[.!?])\s+", desc) if TARGET_SENT.search(s)]
    text = " ".join(sents)
    found = _match(REGION_PATTERNS, text) + _match(SECTOR_PATTERNS, text)
    return ", ".join(found) if found else "Unknown"

d = clean_text(groups.loc[groups["name"] == "APT-C-36", "description"].iloc[0])
print(infer_object(d), "|", infer_target_area(d))

Cyber Espionage and Financial | Latin America, Government, Financial, Energy, Manufacturing


In [ ]:
# ----------------------------- ساخت موجودیت‌ها (گروه‌ها / کمپین‌ها) -----------------------------
uses = rels[rels["mapping type"] == "uses"]

grp_tech  = uses[(uses["source type"] == "group")    & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()
camp_tech = uses[(uses["source type"] == "campaign") & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()
grp_sw    = uses[(uses["source type"] == "group")    & (uses["target type"] == "software")].groupby("source ID")["target ID"].apply(set).to_dict()
sw_tech   = uses[(uses["source type"] == "software") & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()

def make_entities() -> list:
    ents = []
    if INCLUDE_GROUPS:
        for _, row in groups.iterrows():
            techs = set(grp_tech.get(row["ID"], set()))
            if INCLUDE_SOFTWARE_TECHNIQUES:
                for s in grp_sw.get(row["ID"], set()):
                    techs |= sw_tech.get(s, set())
            ents.append(dict(id=row["ID"], name=row["name"], kind="group", techniques=techs, description=clean_text(row["description"])))
    if INCLUDE_ATTACK_CAMPAIGNS:
        for _, row in campaigns.iterrows():
            ents.append(dict(id=row["ID"], name=row["name"], kind="campaign", techniques=set(camp_tech.get(row["ID"], set())), description=clean_text(row["description"])))
    # یکتا کردن نام‌ها
    cnt = collections.Counter(e["name"] for e in ents)
    for e in ents:
        if cnt[e["name"]] > 1:
            e["name"] = f'{e["name"]} ({e["id"]})'
    return ents

ENTITIES = make_entities()
print(len(ENTITIES), collections.Counter(e["kind"] for e in ENTITIES))

232 Counter({'group': 176, 'campaign': 56})


In [ ]:
# ----------------------------- CampaignSequenceBuilder -----------------------------
class CampaignSequenceBuilder:
    """ساخت نمونه‌های next-technique با واحد tactic_rank (ورودی تجمعی یا پنجرهٔ لغزان)."""
    NAME, VERSION = "CampaignSequenceBuilder", "1.0"

    def __init__(self, technique_tactics, window_size=None, min_distinct_ranks=2):
        self.tt = technique_tactics
        self.window = window_size
        self.min_ranks = min_distinct_ranks
        self.unknown_techniques = collections.Counter()

    def _items_by_rank(self, techs):
        by_rank = collections.defaultdict(list)
        for t in sorted(techs):
            if t not in self.tt:
                self.unknown_techniques[t] += 1
                continue
            for rank, _ in self.tt[t]:
                by_rank[rank].append(t)
        return by_rank

    def build_campaign(self, ent):
        by_rank = self._items_by_rank(ent["techniques"])
        ranks = sorted(by_rank)
        if len(ranks) < self.min_ranks:
            return None                      # کمپین نامعتبر
        obj  = infer_object(ent["description"])
        area = infer_target_area(ent["description"])
        history, seen, samples = [], set(), []
        for i, rk in enumerate(ranks):
            if i > 0:
                prev = ranks[i - 1]
                inp = history[-self.window:] if self.window else list(history)
                for tgt in by_rank[rk]:
                    if tgt in seen:          # تکنیکی که قبلاً در تاریخچه بوده هدف نمی‌شود
                        continue
                    samples.append({
                        "campaign": ent["name"],
                        "input": list(inp),
                        "target": tgt,
                        "input_tactic_rank": prev,
                        "target_tactic_rank": rk,
                        "input_tactic": TACTIC_ORDER[prev],
                        "target_tactic": TACTIC_ORDER[rk],
                        "object": obj,
                        "target_area": area,
                    })
            for t in by_rank[rk]:
                if t not in seen:
                    seen.add(t); history.append(t)
        return samples or None

    def build(self, entities, source_name):
        samples, valid, skipped = [], 0, 0
        for ent in entities:
            res = self.build_campaign(ent)
            if res is None:
                skipped += 1
            else:
                valid += 1; samples.extend(res)
        unique = len({(tuple(s["input"]), s["target"]) for s in samples})
        meta = {
            "builder": self.NAME, "version": self.VERSION,
            "source": source_name,
            "task": "next_technique_prediction",
            "sequence_unit": "tactic_rank",
            "total_campaigns": len(entities),
            "valid_campaigns": valid,
            "skipped_campaigns": skipped,
            "total_samples": len(samples),
            "unique_samples": unique,
            "config": {
                "window_size": self.window, "multi_tactic_policy": MULTI_TACTIC_POLICY,
                "include_groups": INCLUDE_GROUPS, "include_attack_campaigns": INCLUDE_ATTACK_CAMPAIGNS,
                "include_software_techniques": INCLUDE_SOFTWARE_TECHNIQUES,
                "min_distinct_ranks": self.min_ranks, "tactic_rank": TACTIC_RANK,
            },
        }
        return {"metadata": meta, "samples": samples}

builder = CampaignSequenceBuilder(TECH_TACTICS, window_size=WINDOW_SIZE, min_distinct_ranks=MIN_DISTINCT_RANKS)
dataset = builder.build(ENTITIES, Path(XLSX_PATH).name)
print(json.dumps(dataset["metadata"], indent=2, ensure_ascii=False)[:900])
print("unknown techniques (not in sheet/tactic):", len(builder.unknown_techniques))

{
  "builder": "CampaignSequenceBuilder",
  "version": "1.0",
  "source": "enterprise-attack-v19_2.xlsx",
  "task": "next_technique_prediction",
  "sequence_unit": "tactic_rank",
  "total_campaigns": 232,
  "valid_campaigns": 221,
  "skipped_campaigns": 11,
  "total_samples": 5232,
  "unique_samples": 5130,
  "config": {
    "window_size": null,
    "multi_tactic_policy": "first",
    "include_groups": true,
    "include_attack_campaigns": true,
    "include_software_techniques": false,
    "min_distinct_ranks": 2,
    "tactic_rank": {
      "reconnaissance": 0,
      "resource-development": 1,
      "initial-access": 2,
      "execution": 3,
      "persistence": 4,
      "privilege-escalation": 5,
      "stealth": 6,
      "defense-impairment": 7,
      "credential-access": 8,
      "discovery": 9,
      "lateral-movement": 10,
      "collection": 11,
      "command-and-control": 12,
  
unknown techniques (not in sheet/tactic): 0


In [ ]:
# ----------------------------- ذخیرهٔ خروجی -----------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print("saved:", OUTPUT_PATH, f"({OUTPUT_PATH.stat().st_size/1e6:.2f} MB)")

saved: Apt-sequence.json (4.01 MB)


In [ ]:
# ----------------------------- تست‌های اعتبارسنجی -----------------------------
with open(OUTPUT_PATH, encoding="utf-8") as f:
    ds = json.load(f)
meta, S = ds["metadata"], ds["samples"]

KEYS = ["campaign","input","target","input_tactic_rank","target_tactic_rank","input_tactic","target_tactic","object","target_area"]
assert meta["total_samples"] == len(S) > 0
assert meta["valid_campaigns"] + meta["skipped_campaigns"] == meta["total_campaigns"]
assert meta["unique_samples"] <= meta["total_samples"]
for s in S:
    assert list(s.keys()) == KEYS
    assert s["input"] and s["target"] not in s["input"]
    assert s["input_tactic_rank"] < s["target_tactic_rank"]
    assert TACTIC_ORDER[s["input_tactic_rank"]] == s["input_tactic"]
    assert TACTIC_ORDER[s["target_tactic_rank"]] == s["target_tactic"]
    assert len(s["input"]) == len(set(s["input"]))
    if WINDOW_SIZE: assert len(s["input"]) <= WINDOW_SIZE
    if MULTI_TACTIC_POLICY == "first":
        assert TECH_TACTICS[s["target"]][0][0] == s["target_tactic_rank"]
        assert TECH_TACTICS[s["input"][-1]][0][0] == s["input_tactic_rank"]   # آخرین ورودی در رتبهٔ ورودی است
        ranks_in = [TECH_TACTICS[t][0][0] for t in s["input"]]
        assert ranks_in == sorted(ranks_in)                                    # ورودی به‌ترتیب رتبه
print("✓ all structural checks passed on", len(S), "samples")

# بررسی نمونه‌ای مشابه قالب ارسالی
ex = next(s for s in S if s["campaign"] == "APT-C-36")
print(json.dumps(ex, indent=2, ensure_ascii=False))

✓ all structural checks passed on 5232 samples
{
  "campaign": "APT-C-36",
  "input": [
    "T1593"
  ],
  "target": "T1583.001",
  "input_tactic_rank": 0,
  "target_tactic_rank": 1,
  "input_tactic": "reconnaissance",
  "target_tactic": "resource-development",
  "object": "Cyber Espionage and Financial",
  "target_area": "Latin America, Government, Financial, Energy, Manufacturing"
}


In [ ]:
# ----------------------------- آمار توصیفی -----------------------------
df = pd.DataFrame(S)
print("campaigns with samples:", df["campaign"].nunique())
print("\nsamples per target tactic:")
print(df["target_tactic"].value_counts().reindex(TACTIC_ORDER).dropna().astype(int).to_string())
print("\ninput length (min/median/max):", df["input"].map(len).min(), df["input"].map(len).median(), df["input"].map(len).max())
print("\nobject == Unknown     :", f'{(df["object"]=="Unknown").mean():.1%}')
print("target_area == Unknown:", f'{(df["target_area"]=="Unknown").mean():.1%}')
print("\ntop-5 campaigns by #samples:")
print(df["campaign"].value_counts().head(5).to_string())

campaigns with samples: 221

samples per target tactic:
target_tactic
resource-development    299
initial-access          395
execution               821
persistence             327
privilege-escalation    107
stealth                 730
defense-impairment      154
credential-access       366
discovery               734
lateral-movement        196
collection              338
command-and-control     544
exfiltration            118
impact                  103

input length (min/median/max): 1 16.0 128

object == Unknown     : 31.7%
target_area == Unknown: 17.5%

top-5 campaigns by #samples:
campaign
Kimsuky          120
Lazarus Group     91
APT28             87
Mustang Panda     83
APT41             79
